# Ground-Truth Evaluation and Baselines for the Cellpose Nanoparticle Pipeline

This notebook adds what reviewers asked for: **proof that the segmentation and sizes are correct**, and **comparison with other methods**.

**Your steps (only 3):**
1. Run **Step 0–1**. It saves preprocessed copies of your SEM images to the `annotate/` folder.
2. Open 3 of those PNG images in **Fiji/ImageJ**, outline every particle, and save the outlines (instructions in Step 2).
3. Run **Step 3–4**. It compares Cellpose, Otsu + watershed and StarDist against your hand outlines, and prints the tables for the paper.

Choose 3 images: one **sparse**, one **medium** and one **dense** (include Image 12).

## Step 0 · Install and settings

In [ ]:
!pip -q install "cellpose>=4" stardist roifile tifffile
# If StarDist fails to install, the notebook still runs Cellpose and Otsu.

In [ ]:
# ---- EDIT THESE ----
IMAGE_DIR   = "/content/sem_images"     # folder with the 15 Zeiss .tif images
OUT_DIR     = "/content/eval_out"       # results are written here
ANNOT_DIR   = f"{OUT_DIR}/annotate"     # preprocessed PNGs to annotate in Fiji
GT_DIR      = f"{OUT_DIR}/ground_truth" # put your Fiji RoiSet .zip files here

CROP_BOTTOM_FRAC = 0.0   # use the SAME data-bar crop as your main notebook (e.g. 0.07)
MAX_EDGE         = 1500  # same as the paper
MIN_AREA_PX      = 100   # same as the paper
IOU_THRESHOLD    = 0.5   # a predicted particle counts as correct if IoU >= 0.5

# Only if an image has no Zeiss metadata: {"filename.tif": nm_per_pixel}
MANUAL_NM_PER_PX = {}

import os
for d in (OUT_DIR, ANNOT_DIR, GT_DIR):
    os.makedirs(d, exist_ok=True)
print("Folders ready:", OUT_DIR)

## Core functions (run once, no need to edit)

In [ ]:
# ---------- CORE FUNCTIONS (preprocessing, ground truth, methods, matching) ----------
import os, glob, re, time
import numpy as np
import pandas as pd
import tifffile
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment
from skimage import io, exposure, filters, measure, morphology, segmentation, transform, draw, util
from skimage.feature import peak_local_max


# ---------- calibration ----------
def read_nm_per_px(path, manual=None):
    """nm per pixel from Zeiss CZ_SEM metadata (tag 34118); manual dict overrides."""
    name = os.path.basename(path)
    if manual and name in manual:
        return float(manual[name]), "manual"
    try:
        with tifffile.TiffFile(path) as tf:
            tag = tf.pages[0].tags.get(34118)
            if tag is None:
                return None, "missing"
            meta = tag.value
            items = meta.items() if isinstance(meta, dict) else []
            for key, val in items:
                if str(key).lower() in ("ap_image_pixel_size", "ap_pixel_size"):
                    # tifffile gives e.g. ('Image Pixel Size', 1.86, 'nm')
                    num, unit = None, "nm"
                    if isinstance(val, (tuple, list)):
                        for v in val:
                            if isinstance(v, (int, float)) and num is None:
                                num = float(v)
                            elif isinstance(v, str) and v.strip().lower() in ("pm", "nm", "um", "µm", "mm"):
                                unit = v.strip().lower()
                    else:
                        m = re.search(r"([\d.]+)\s*(pm|nm|um|µm|mm)?", str(val))
                        if m:
                            num, unit = float(m.group(1)), (m.group(2) or "nm").lower()
                    if num is not None:
                        factor = {"pm": 1e-3, "nm": 1.0, "um": 1e3, "µm": 1e3, "mm": 1e6}[unit]
                        return num * factor, "metadata"
    except Exception as e:
        print("  metadata read failed for", name, ":", e)
    return None, "missing"


# ---------- preprocessing (same steps as the paper) ----------
def preprocess(path, nm_per_px, crop_bottom_frac=0.0, max_edge=1500):
    img = io.imread(path)
    if img.ndim == 3:
        img = img[..., :3].mean(axis=2)
    img = util.img_as_float(img.astype(np.float32) / (img.max() if img.max() > 0 else 1))
    if crop_bottom_frac > 0:
        img = img[: int(round(img.shape[0] * (1 - crop_bottom_frac))), :]
    img = exposure.equalize_hist(img)
    scale = 1.0
    if max(img.shape) > max_edge:
        scale = max_edge / max(img.shape)
        img = transform.rescale(img, scale, anti_aliasing=True, preserve_range=True)
    return img.astype(np.float32), nm_per_px / scale


# ---------- ground truth ----------
def load_ground_truth(gt_path, shape):
    """Fiji RoiSet.zip (needs `roifile`) or a label-mask TIFF/PNG -> label image."""
    if gt_path.lower().endswith(".zip"):
        import roifile
        lab = np.zeros(shape, dtype=np.int32)
        rois = roifile.roiread(gt_path)
        if not isinstance(rois, list):
            rois = [rois]
        k = 0
        for r in rois:
            pts = r.coordinates()
            if pts is None or len(pts) < 3:
                continue
            rr, cc = draw.polygon(pts[:, 1], pts[:, 0], shape)
            k += 1
            lab[rr, cc] = k
        return lab
    lab = io.imread(gt_path)
    if lab.ndim == 3:
        lab = lab[..., 0]
    return measure.label(lab > 0) if lab.max() == 1 or lab.dtype == bool else lab.astype(np.int32)


# ---------- segmentation methods ----------
def remove_small(lab, min_area_px):
    """Drop objects smaller than min_area_px (works on old and new scikit-image)."""
    lab = measure.label(lab > 0) if lab.dtype == bool else lab
    areas = np.bincount(lab.ravel())
    small = np.where(areas < min_area_px)[0]
    out = lab.copy()
    out[np.isin(out, small[small > 0])] = 0
    return segmentation.relabel_sequential(out)[0]


def watershed_refine(mask, min_area_px=100, h=2.0):
    """Split touching objects: distance transform + h-maxima markers (paper's step 3)."""
    fg = mask > 0
    dist = ndi.gaussian_filter(ndi.distance_transform_edt(fg), 1.0)
    markers = measure.label(morphology.h_maxima(dist, h))
    lab = segmentation.watershed(-dist, markers, mask=fg)
    return remove_small(lab, min_area_px)


def seg_otsu_watershed(img, min_area_px=100):
    sm = filters.gaussian(img, 1.5)
    fg = sm > filters.threshold_otsu(sm)
    fg = ndi.binary_opening(ndi.binary_fill_holes(fg), structure=morphology.disk(2))
    fg = remove_small(measure.label(fg), min_area_px) > 0
    return watershed_refine(fg.astype(np.int32), min_area_px)


_CP_MODEL = None
def seg_cellpose(img, min_area_px=100, refine=True):
    global _CP_MODEL
    from cellpose import models
    if _CP_MODEL is None:
        try:
            _CP_MODEL = models.CellposeModel(gpu=True)          # Cellpose v4 (default model)
        except TypeError:
            _CP_MODEL = models.CellposeModel(gpu=True, model_type="cyto")
    out = _CP_MODEL.eval(img, diameter=None, flow_threshold=0.4, cellprob_threshold=0.2)
    masks = out[0]
    masks = remove_small(masks.astype(np.int32), min_area_px)
    return watershed_refine(masks > 0, min_area_px) if refine else masks


_SD_MODEL = None
def seg_stardist(img, min_area_px=100):
    global _SD_MODEL
    from stardist.models import StarDist2D
    from csbdeep.utils import normalize
    if _SD_MODEL is None:
        _SD_MODEL = StarDist2D.from_pretrained("2D_versatile_fluo")
    lab, _ = _SD_MODEL.predict_instances(normalize(img, 1, 99.8))
    return remove_small(lab.astype(np.int32), min_area_px)


# ---------- matching and metrics ----------
def iou_matrix(gt, pred):
    g, p = gt.ravel().astype(np.int64), pred.ravel().astype(np.int64)
    ng, npred = int(g.max()), int(p.max())
    inter = np.bincount(g * (npred + 1) + p, minlength=(ng + 1) * (npred + 1)).reshape(ng + 1, npred + 1)
    area_g, area_p = inter.sum(1), inter.sum(0)
    inter = inter[1:, 1:]
    union = area_g[1:, None] + area_p[None, 1:] - inter
    with np.errstate(divide="ignore", invalid="ignore"):
        iou = np.where(union > 0, inter / union, 0.0)
    return iou, area_g[1:], area_p[1:]


def evaluate(gt, pred, nm_per_px, iou_thr=0.5):
    """Per-particle matching (IoU >= thr, one-to-one) and size errors in nm / nm^2."""
    gt = measure.label(gt > 0) if gt.max() <= 1 else gt
    gt, _, _ = segmentation.relabel_sequential(gt)
    pred, _, _ = segmentation.relabel_sequential(pred)
    n_gt, n_pred = int(gt.max()), int(pred.max())
    if n_gt == 0 or n_pred == 0:
        return dict(n_gt=n_gt, n_pred=n_pred, TP=0, FP=n_pred, FN=n_gt, precision=0, recall=0, F1=0), pd.DataFrame()
    iou, ag, ap = iou_matrix(gt, pred)
    cost = np.where(iou >= iou_thr, -iou, 0.0)
    r, c = linear_sum_assignment(cost)
    keep = iou[r, c] >= iou_thr
    r, c = r[keep], c[keep]
    tp = len(r); fp = n_pred - tp; fn = n_gt - tp
    prec = tp / n_pred; rec = tp / n_gt
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    a_nm2 = nm_per_px ** 2
    gt_area, pr_area = ag[r] * a_nm2, ap[c] * a_nm2
    gt_d, pr_d = 2 * np.sqrt(gt_area / np.pi), 2 * np.sqrt(pr_area / np.pi)
    pairs = pd.DataFrame(dict(gt_id=r + 1, pred_id=c + 1, IoU=iou[r, c],
                              gt_diam_nm=gt_d, pred_diam_nm=pr_d,
                              gt_area_nm2=gt_area, pred_area_nm2=pr_area))
    summary = dict(
        n_gt=n_gt, n_pred=n_pred, TP=tp, FP=fp, FN=fn,
        precision=prec, recall=rec, F1=f1,
        mean_IoU=float(iou[r, c].mean()) if tp else 0.0,
        count_error_pct=100 * (n_pred - n_gt) / n_gt,
        diam_MAE_nm=float(np.abs(pr_d - gt_d).mean()) if tp else np.nan,
        diam_bias_nm=float((pr_d - gt_d).mean()) if tp else np.nan,
        diam_MAPE_pct=float((100 * np.abs(pr_d - gt_d) / gt_d).mean()) if tp else np.nan,
        area_MAPE_pct=float((100 * np.abs(pr_area - gt_area) / gt_area).mean()) if tp else np.nan,
        gt_mean_diam_nm=float((2 * np.sqrt(ag * a_nm2 / np.pi)).mean()),
        pred_mean_diam_nm=float((2 * np.sqrt(ap * a_nm2 / np.pi)).mean()),
    )
    return summary, pairs


## Step 1 · Preprocess images and save copies for annotation

In [ ]:
rows = []
files = sorted(glob.glob(os.path.join(IMAGE_DIR, "*.tif")) + glob.glob(os.path.join(IMAGE_DIR, "*.tiff")))
assert files, f"No .tif files found in {IMAGE_DIR}"
for i, f in enumerate(files, 1):
    nmpp, src = read_nm_per_px(f, MANUAL_NM_PER_PX)
    if nmpp is None:
        print(f"STOP: no calibration for {os.path.basename(f)} - add it to MANUAL_NM_PER_PX"); continue
    img, nmpp_scaled = preprocess(f, nmpp, CROP_BOTTOM_FRAC, MAX_EDGE)
    name = f"Image_{i:02d}"
    io.imsave(os.path.join(ANNOT_DIR, name + ".png"), (img * 255).astype(np.uint8), check_contrast=False)
    rows.append(dict(image=name, source_file=os.path.basename(f), nm_per_px_native=nmpp,
                     nm_per_px_used=nmpp_scaled, calibration=src, height=img.shape[0], width=img.shape[1]))
calib = pd.DataFrame(rows)
calib.to_csv(os.path.join(OUT_DIR, "calibration.csv"), index=False)
calib

## Step 2 · Annotate 3 images in Fiji (you do this)

For each chosen image in `annotate/` (for example `Image_03.png`, `Image_07.png`, `Image_12.png`):

1. Open the PNG in **Fiji** (*File → Open*).
2. Choose the **Freehand** or **Oval** selection tool.
3. Draw around **one particle**, then press **T** to add it to the *ROI Manager*. Repeat for **every** particle.
   - Draw along the particle's visible edge.
   - Skip particles cut off by the image border, and do the same for all methods (this is already handled).
4. In the ROI Manager: *More → Save…* and save as **`Image_03_RoiSet.zip`** (image name + `_RoiSet.zip`).
5. Upload the zip files to the `ground_truth/` folder.

**Tip:** for the dense Image 12, annotating a cropped region is fine. Crop the PNG in Fiji first (*Image → Crop*), save it as `Image_12.png` in `annotate/` (replacing the original), then annotate that.

About 30–60 minutes per image.

## Step 3 · Run all methods and compare with your outlines

In [ ]:
METHODS = {
    "Cellpose + watershed (ours)": lambda im: seg_cellpose(im, MIN_AREA_PX, refine=True),
    "Otsu + watershed":            lambda im: seg_otsu_watershed(im, MIN_AREA_PX),
    "StarDist (pretrained)":       lambda im: seg_stardist(im, MIN_AREA_PX),
}

calib = pd.read_csv(os.path.join(OUT_DIR, "calibration.csv")).set_index("image")
results, all_pairs, preds_cache = [], [], {}
gt_files = sorted(glob.glob(os.path.join(GT_DIR, "*_RoiSet.zip")) + glob.glob(os.path.join(GT_DIR, "*_mask.tif")))
assert gt_files, "No ground-truth files yet - finish Step 2 first."

for gtf in gt_files:
    name = re.sub(r"_(RoiSet\.zip|mask\.tif)$", "", os.path.basename(gtf))
    img = io.imread(os.path.join(ANNOT_DIR, name + ".png")).astype(np.float32) / 255.0
    if img.ndim == 3: img = img[..., 0]
    nmpp = float(calib.loc[name, "nm_per_px_used"])
    gt = load_ground_truth(gtf, img.shape)
    print(f"{name}: {gt.max()} hand-annotated particles")
    for mname, fn in METHODS.items():
        try:
            t0 = time.time(); pred = fn(img); dt = time.time() - t0
        except Exception as e:
            print(f"   {mname}: skipped ({e})"); continue
        s, pairs = evaluate(gt, pred, nmpp, IOU_THRESHOLD)
        s.update(image=name, method=mname, time_s=dt); results.append(s)
        pairs["image"], pairs["method"] = name, mname; all_pairs.append(pairs)
        preds_cache[(name, mname)] = pred
        print(f"   {mname:30s} F1={s['F1']:.3f}  diam MAE={s['diam_MAE_nm']:.2f} nm  bias={s['diam_bias_nm']:+.2f} nm")

per_image = pd.DataFrame(results)
pairs_df = pd.concat(all_pairs, ignore_index=True) if all_pairs else pd.DataFrame()
per_image.to_csv(os.path.join(OUT_DIR, "per_image_results.csv"), index=False)
pairs_df.to_csv(os.path.join(OUT_DIR, "matched_particles.csv"), index=False)
print("Saved per_image_results.csv and matched_particles.csv")

## Step 4 · Tables and figures for the paper

In [ ]:
# Pooled over all annotated images (sum TP/FP/FN, then compute metrics)
def pooled(g):
    tp, fp, fn = g.TP.sum(), g.FP.sum(), g.FN.sum()
    p = tp / (tp + fp) if tp + fp else 0; r = tp / (tp + fn) if tp + fn else 0
    pr = pairs_df[pairs_df.method == g.name]
    err = pr.pred_diam_nm - pr.gt_diam_nm
    return pd.Series({
        "Annotated particles": int(g.n_gt.sum()), "Detected": int(g.n_pred.sum()),
        "Precision": p, "Recall": r, "F1": 2*p*r/(p+r) if p+r else 0,
        "Mean IoU": pr.IoU.mean(),
        "Diameter MAE (nm)": err.abs().mean(), "Diameter bias (nm)": err.mean(),
        "Area error (%)": (100*(pr.pred_area_nm2 - pr.gt_area_nm2).abs()/pr.gt_area_nm2).mean(),
        "Time per image (s)": g.time_s.mean()})

table = per_image.groupby("method").apply(pooled).round(3)
table = table.sort_values("F1", ascending=False)
table.to_csv(os.path.join(OUT_DIR, "TABLE_comparison.csv"))
display(table)
print(table.to_markdown())

In [ ]:
import matplotlib.pyplot as plt
# Figure A: predicted vs hand-measured diameter (matched particles), one panel per method
ms = list(table.index)
fig, axes = plt.subplots(1, len(ms), figsize=(5*len(ms), 4.5), squeeze=False)
for ax, m in zip(axes[0], ms):
    d = pairs_df[pairs_df.method == m]
    ax.scatter(d.gt_diam_nm, d.pred_diam_nm, s=8, alpha=0.5)
    lo, hi = d[["gt_diam_nm", "pred_diam_nm"]].min().min(), d[["gt_diam_nm", "pred_diam_nm"]].max().max()
    ax.plot([lo, hi], [lo, hi], "k--", lw=1, label="y = x")
    ax.set_title(f"{m}\nMAE {table.loc[m,'Diameter MAE (nm)']:.1f} nm, bias {table.loc[m,'Diameter bias (nm)']:+.1f} nm", fontsize=10)
    ax.set_xlabel("Hand-measured diameter (nm)"); ax.set_ylabel("Predicted diameter (nm)"); ax.legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "FIG_diameter_agreement.png"), dpi=300); plt.show()

# Figure B: F1 per image and method
piv = per_image.pivot(index="image", columns="method", values="F1")
piv.plot(kind="bar", figsize=(7, 4), rot=0); plt.ylabel("F1 (IoU ≥ 0.5)"); plt.ylim(0, 1)
plt.title("Detection accuracy against hand annotation"); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "FIG_F1_per_image.png"), dpi=300); plt.show()

# Figure C: overlay for one image: green = correct, red = extra (FP), yellow = missed (FN)
from skimage.segmentation import find_boundaries
name = per_image.image.iloc[0]
m = "Cellpose + watershed (ours)" if (name, "Cellpose + watershed (ours)") in preds_cache else per_image.method.iloc[0]
if (name, m) in preds_cache:
    img = io.imread(os.path.join(ANNOT_DIR, name + ".png")).astype(float) / 255
    gt = load_ground_truth([g for g in gt_files if os.path.basename(g).startswith(name)][0], img.shape)
    pred = preds_cache[(name, m)]
    pr = pairs_df[(pairs_df.image == name) & (pairs_df.method == m)]
    rgb = np.dstack([img] * 3)
    tp_mask = np.isin(pred, pr.pred_id.values); fp_mask = (pred > 0) & ~tp_mask
    fn_mask = (gt > 0) & ~np.isin(gt, pr.gt_id.values)
    for msk, col in ((tp_mask, (0, 1, 0)), (fp_mask, (1, 0, 0)), (fn_mask, (1, 1, 0))):
        rgb[find_boundaries(np.where(msk, pred if msk is not fn_mask else gt, 0))] = col
    plt.figure(figsize=(8, 6)); plt.imshow(rgb); plt.axis("off")
    plt.title(f"{name}: green = correct, red = extra, yellow = missed")
    plt.savefig(os.path.join(OUT_DIR, f"FIG_overlay_{name}.png"), dpi=300, bbox_inches="tight"); plt.show()

In [ ]:
# Ready-to-paste sentence for the Results section
key = "Cellpose + watershed (ours)" if "Cellpose + watershed (ours)" in table.index else table.index[0]
ours = table.loc[key]
print(f"""Against hand annotation of {int(ours['Annotated particles'])} particles in {per_image.image.nunique()} SEM images, \
the proposed pipeline achieved precision {ours['Precision']:.2f}, recall {ours['Recall']:.2f} and F1 {ours['F1']:.2f} \
(IoU ≥ {IOU_THRESHOLD}), with a mean absolute diameter error of {ours['Diameter MAE (nm)']:.1f} nm \
(bias {ours['Diameter bias (nm)']:+.1f} nm) and a mean projected-area error of {ours['Area error (%)']:.1f}%.""")